> 在线讲义：[第 16 章 · 多模态与实时交互：数据换形状，时间有死线（略讲）](https://wikieden.github.io/ai-infra-from-constraints/lessons/16-multimodal-realtime.html) · 本书《从约束推导 AI Infra》

# 第 16 章 · 多模态与实时交互：数据换形状，时间有死线（略讲）

配套讲义：仓库里 `ai-infra/notes/lessons/16-multimodal-realtime.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 视觉 token | $HW/(p^2r^2)$ | `vision_tokens(640, 640)` | 400 |
| 第一步 三种大小 | 乘宽度与字节 | `vision_sizes(400)` | EC 7.8125 MiB，KV 56.25 MiB；四图 + 文本 281.25 MiB |
| 第二步 图像生成 | 步数 × 分支 | `dit_forwards(50, 2)` | 100 次 |
| 第三步 每块字节 | $24000 \times 0.02 \times 2$ | `chunk_bytes()` | 960 |
| 第三步 播放时间线 | 逐块模拟 | `playback(40, {2: 50})`、`playback(60, {2: 50})` | 首播 78 ms、停顿 5 ms；首播 98 ms、停顿 0 |

Qwen3-VL-4B 的视觉 token 与特征宽度、Qwen-Image 的 50 步 × 2 路 CFG、20 ms 音频块的教学时间线取自 AII 第 3 章 §3.3。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `vision_tokens`**：视觉 token 数 n_v = H·W / (p² r²)。

In [ ]:
def vision_tokens(h: int, w: int, patch: int = 16, merge: int = 2) -> int:
    """视觉 token 数 n_v = H·W / (p² r²)。"""
    return h * w // (patch**2 * merge**2)

**函数 `vision_sizes`**：(完整 EC MiB, 视觉 token 的语言 KV MiB)。EC = n_v × (groups × width) × b。

In [ ]:
def vision_sizes(n_v: int, width: int = 2560, groups: int = 4, b: int = 2, kv_kib: int = 144) -> tuple[float, float]:
    """(完整 EC MiB, 视觉 token 的语言 KV MiB)。EC = n_v × (groups × width) × b。"""
    return n_v * groups * width * b / 2**20, n_v * kv_kib / 1024

**函数 `dit_forwards`**：图像生成的 DiT 前向次数 = 去噪步数 × 引导分支数。

In [ ]:
def dit_forwards(steps: int, branches: int) -> int:
    """图像生成的 DiT 前向次数 = 去噪步数 × 引导分支数。"""
    return steps * branches

**函数 `chunk_bytes`**：一块音频的字节数 = 采样率 × 时长 × 声道 × 每样本字节。

In [ ]:
def chunk_bytes(rate: int = 24_000, ms: int = 20, channels: int = 1, bytes_per_sample: int = 2) -> int:
    """一块音频的字节数 = 采样率 × 时长 × 声道 × 每样本字节。"""
    return rate * ms // 1000 * channels * bytes_per_sample

**函数 `playback`**：逐块模拟：(首播时刻 ms, 总停顿 ms)。第 i 块在 20(i+1) ms 采集完，处理 + 发送 13 ms，传播默认 5 ms。

In [ ]:
def playback(buf_ms: int, prop_override: dict[int, int], n: int = 8, chunk: int = 20, proc: int = 13, prop: int = 5) -> tuple[int, int]:
    """逐块模拟：(首播时刻 ms, 总停顿 ms)。第 i 块在 20(i+1) ms 采集完，处理 + 发送 13 ms，传播默认 5 ms。"""
    arrive = [chunk * (i + 1) + proc + prop_override.get(i, prop) for i in range(n)]
    t = arrive[0] + buf_ms  # 第一块到达后再等 buf_ms 开播
    start, stall = t, 0
    for i in range(1, n):
        t += chunk  # 上一块播完，下一块的计划时刻
        if arrive[i] > t:  # 没到：停顿、顺延
            stall += arrive[i] - t
            t = arrive[i]
    return start, stall

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：Qwen3-VL-4B，640×640")
n_v = vision_tokens(640, 640)
ec, kv = vision_sizes(n_v)
print(f"  {n_v} 个视觉 token；EC {ec:.4f} MiB；语言 KV {kv:.2f} MiB")
print(f"  四张图 + 400 文本 token：EC {4 * ec:.2f} MiB，输入 KV {4 * kv + 400 * 144 / 1024:.2f} MiB")

In [ ]:
print("\n第二步：Qwen-Image，50 步 × 2 路 CFG")
print(f"  {dit_forwards(50, 2)} 次 DiT 前向")

In [ ]:
print("\n第三步：20 ms 一块音频")
print(f"  每块 {chunk_bytes()} 字节")
for buf in (40, 60):
    start, stall = playback(buf, {2: 50})
    print(f"  缓冲 {buf} ms：首播 {start} ms，停顿 {stall} ms")

## 3. 代码里才看得到的

1. `proc=13` 是"处理 12 ms + 发送 1 ms"；第 $i$ 块在 $20(i+1)$ ms 采集完，所以第一块到达是 $20 + 13 + 5 = 38$ ms。
2. 停顿发生后，后面所有块的播放都顺延（`t = arrive[i]`），这就是原文"后续块虽已到达，也要按声音顺序等待"。
3. EC 的 7.8125 MiB 是四组 2560 维特征拼起来的 $[400, 10240]$；只看最终层是 1.953 MiB。

## 4. 自己改一改

- 把 `playback` 的第三块延迟改成 30、90 ms，看 40 ms 缓冲还停不停。
- 把 `vision_tokens` 的合并 $r$ 从 2 改成 1，看视觉 token 和 KV 涨多少。
- 让每块都随机多 0–20 ms 延迟，多跑几次，找出不停顿所需的最小缓冲。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 地图切格再合卡，张数是位置，页宽是数据；
> 原图、EC、KV，三站三种大小；
> 画画一遍遍修，步数乘分支乘单步；
> 桶里有水不断流，桶越满开始越晚；
> 闭麦、停排练、撤道具，三件事三个钟。